In [23]:
import numpy as np
import pandas as pd
import os
import shutil
import matplotlib.pyplot as plt
import datetime
import cv2 as cv
from sklearn.metrics import r2_score
from scipy.stats import pearsonr,spearmanr
import matplotlib.dates as mdates
import datetime
from sklearn.decomposition import PCA, KernelPCA
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, GridSearchCV
import joblib as jb
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, AdaBoostRegressor, BaggingRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.cluster import SpectralClustering, MiniBatchKMeans, SpectralBiclustering, SpectralCoclustering
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import Ridge, Lasso
import cv2 as cv
from skimage.metrics import structural_similarity
import matplotlib.dates as mdates



In [24]:
#INPUT CSV DATA
ghusuri=pd.read_csv('ghusuri.csv')
ghusuri_frb=pd.read_csv('ghusuri_frb.csv')

In [25]:
#EXTRACTING GHUSURI DATA
ghusuri['Day'] = pd.to_numeric(ghusuri['ImageFile'].str[9:11])
ghusuri['Month'] = ghusuri['ImageFile'].str[5:8]
ghusuri['Hour'] =pd.to_numeric(ghusuri['ImageFile'].str[12:14])
ghusuri['Min'] = pd.to_numeric(ghusuri['ImageFile'].str[15:17])

In [26]:
#EXTRACTING GHUSURI_RBU DATA
ghusuri_frb['Day'] = pd.to_numeric(ghusuri_frb['ImageFile'].str[9:11])
ghusuri_frb['Month'] = ghusuri_frb['ImageFile'].str[5:8]
ghusuri_frb['Hour'] =pd.to_numeric(ghusuri_frb['ImageFile'].str[12:14])
ghusuri_frb['Min'] = pd.to_numeric(ghusuri_frb['ImageFile'].str[15:17])

In [27]:
#TURNING MONTHS INTO NUMBERS
Months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

m =[]
for i,r in ghusuri.iterrows():
    s = str(r['Month'])
    i = Months.index(s)
    m.append(i+1)

ghusuri['Month_num']=m

In [28]:
#CONVERTING GHUSURI DAY,MONTH,HOUR,MIN INTO CYCLIC DATA
ghusuri['Day_sin'] = np.sin(2 * np.pi * ghusuri['Day']/31)
ghusuri['Day_cos'] = np.cos(2 * np.pi * ghusuri['Day']/31)
ghusuri['Mon_sin'] = np.sin(2 * np.pi * ghusuri['Month_num']/12)
ghusuri['Mon_cos'] = np.cos(2 * np.pi * ghusuri['Month_num']/12)
ghusuri['Hour_sin'] = np.sin(2 * np.pi * ghusuri['Hour']/24)
ghusuri['Hour_cos'] = np.cos(2 * np.pi * ghusuri['Hour']/24)
ghusuri['Min_sin'] = np.sin(2 * np.pi * ghusuri['Min']/60)
ghusuri['Min_cos'] = np.cos(2 * np.pi * ghusuri['Min']/60)

In [29]:
m= []
for i,r in ghusuri_frb.iterrows():
    s= str(r["Month"])
    i= Months.index(s)
    m.append(i+1)
ghusuri_frb['Month_num'] = m

In [30]:
#CONVERTING GHUSURI_RBU DAY,MONTH,HOUR,MIN INTO CYCLIC DATA
ghusuri_frb['Day_sin'] = np.sin(2 * np.pi * ghusuri_frb['Day']/31)
ghusuri_frb['Day_cos'] = np.cos(2 * np.pi * ghusuri_frb['Day']/31)
ghusuri_frb['Mon_sin'] = np.sin(2 * np.pi * ghusuri_frb['Month_num']/12)
ghusuri_frb['Mon_cos'] = np.cos(2 * np.pi * ghusuri_frb['Month_num']/12)
ghusuri_frb['Hour_sin'] = np.sin(2 * np.pi * ghusuri_frb['Hour']/24)
ghusuri_frb['Hour_cos'] = np.cos(2 * np.pi * ghusuri_frb['Hour']/24)
ghusuri_frb['Min_sin'] = np.sin(2 * np.pi * ghusuri_frb['Min']/60)
ghusuri_frb['Min_cos'] = np.cos(2 * np.pi * ghusuri_frb['Min']/60)

In [31]:
#SPLIT GHUSURI DATA
train_b,test_b = train_test_split(ghusuri,test_size=0.2,random_state=42)

In [32]:
#SPLIT GHUSURI_RBU DATA
train_bf,test_bf = train_test_split(ghusuri_frb,test_size=0.2,random_state=42)

In [33]:
#NORMALIZING GHUSURI PM2.5 VALUES
train_b['PM2.5_Norm'] = (train_b['PM2.5'] - train_b['PM2.5'].min())/(train_b['PM2.5'].max() - train_b['PM2.5'].min())
test_b['PM2.5_Norm'] = (test_b['PM2.5'] - train_b['PM2.5'].min())/(train_b['PM2.5'].max() - train_b['PM2.5'].min())

In [34]:
#NORMALIZING GHUSURI_RBU PM2.5 VALUES
train_bf['PM2.5_Norm'] = (train_bf['PM2.5'] - train_bf['PM2.5'].min())/(train_bf['PM2.5'].max() - train_bf['PM2.5'].min())
test_bf['PM2.5_Norm'] = (test_bf['PM2.5'] - train_bf['PM2.5'].min())/(train_bf['PM2.5'].max() - train_bf['PM2.5'].min())

In [35]:
#CREATING X1 LIST FOR GHUSURI DATA
X1_train = train_b[['SSIM_ROI','ERODE_SSIM_ROI','Day_sin','Day_cos','Mon_sin','Mon_cos','Hour_sin','Hour_cos']]
X1_test = test_b[['SSIM_ROI','ERODE_SSIM_ROI','Day_sin','Day_cos','Mon_sin','Mon_cos','Hour_sin','Hour_cos']] 


In [36]:
#CREATING X2 LIST FOR GHUSURI_RBU DATA
X2_train = train_bf[['SSIM_ROI','ERODE_SSIM_ROI','Day_sin','Day_cos','Mon_sin','Mon_cos','Hour_sin','Hour_cos']]
X2_test = test_bf[['SSIM_ROI','ERODE_SSIM_ROI','Day_sin','Day_cos','Mon_sin','Mon_cos','Hour_sin','Hour_cos']]

In [37]:
#CREATING Y1 LIST FOR GHUSURI DATA
Y1_train= train_b['PM2.5_Norm']
Y1_test= test_b['PM2.5_Norm']

In [38]:
#CREATING Y2 LIST FOR GHUSURI_RBU DATA
Y2_train= train_bf['PM2.5_Norm']
Y2_test= test_bf['PM2.5_Norm']


In [39]:
#CREATING HYPER_PARAMETER GRID
param_grid = {
    'n_estimators': np.arange(10,1020,10),
    'max_features': ['sqrt','log2',0.1,0.2,0.3,0.5,0.7,1.0],
    'bootstrap': [True,False]
}

In [40]:
#GRID SEARCH FOR GHUSURI DATA
#CREATE GRID SEARCH
grid = GridSearchCV(ExtraTreesRegressor(),param_grid=param_grid,refit=True,verbose=3,n_jobs=-1)

In [41]:

#TRAIN GRID SEARCH
grid.fit(X1_train,Y1_train)

Fitting 5 folds for each of 1616 candidates, totalling 8080 fits
[CV 3/5] END bootstrap=True, max_features=sqrt, n_estimators=20;, score=0.852 total time=   0.0s
[CV 2/5] END bootstrap=True, max_features=sqrt, n_estimators=40;, score=0.692 total time=   0.1s
[CV 1/5] END bootstrap=True, max_features=sqrt, n_estimators=60;, score=0.878 total time=   0.1s
[CV 4/5] END bootstrap=True, max_features=sqrt, n_estimators=70;, score=0.899 total time=   0.1s
[CV 2/5] END bootstrap=True, max_features=sqrt, n_estimators=90;, score=0.565 total time=   0.1s
[CV 5/5] END bootstrap=True, max_features=sqrt, n_estimators=100;, score=0.844 total time=   0.1s
[CV 3/5] END bootstrap=True, max_features=sqrt, n_estimators=120;, score=0.877 total time=   0.2s
[CV 1/5] END bootstrap=True, max_features=sqrt, n_estimators=140;, score=0.883 total time=   0.2s
[CV 4/5] END bootstrap=True, max_features=sqrt, n_estimators=150;, score=0.903 total time=   0.2s
[CV 2/5] END bootstrap=True, max_features=sqrt, n_estimato

/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



[CV 4/5] END bootstrap=True, max_features=log2, n_estimators=300;, score=0.918 total time=   0.5s
[CV 2/5] END bootstrap=True, max_features=log2, n_estimators=320;, score=0.689 total time=   0.5s
[CV 5/5] END bootstrap=True, max_features=log2, n_estimators=330;, score=0.871 total time=   0.5s
[CV 3/5] END bootstrap=True, max_features=log2, n_estimators=350;, score=0.911 total time=   0.5s
[CV 1/5] END bootstrap=True, max_features=log2, n_estimators=370;, score=0.907 total time=   0.6s
[CV 5/5] END bootstrap=True, max_features=log2, n_estimators=380;, score=0.876 total time=   0.6s
[CV 2/5] END bootstrap=True, max_features=log2, n_estimators=400;, score=0.660 total time=   0.6s
[CV 5/5] END bootstrap=True, max_features=log2, n_estimators=410;, score=0.871 total time=   0.6s
[CV 3/5] END bootstrap=True, max_features=log2, n_estimators=430;, score=0.904 total time=   0.7s
[CV 1/5] END bootstrap=True, max_features=log2, n_estimators=450;, score=0.913 total time=   0.7s
[CV 4/5] END bootst

,estimator,ExtraTreesRegressor()
,param_grid,"{'bootstrap': [True, False], 'max_features': ['sqrt', 'log2', ...], 'n_estimators': array([ 10, ... 1000, 1010])}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,np.int64(130)


In [42]:
grid.best_params_

{'bootstrap': False, 'max_features': 1.0, 'n_estimators': np.int64(130)}

In [45]:
grid.score(X1_train,Y1_train)
grid.score(X1_test,Y1_test)

0.9699628136217849

In [46]:
#GRID SEARCH FOR GHUSURI_RBU DATA
#CREATE GRID SEARCH
grid = GridSearchCV(ExtraTreesRegressor(),param_grid=param_grid,refit=True,verbose=3,n_jobs=-1)

In [47]:

#TRAIN GRID SEARCH
grid.fit(X2_train,Y2_train)

Fitting 5 folds for each of 1616 candidates, totalling 8080 fits

[CV 2/5] END bootstrap=False, max_features=0.5, n_estimators=730;, score=0.835 total time=   1.3s
[CV 5/5] END bootstrap=False, max_features=0.5, n_estimators=740;, score=0.933 total time=   1.4s
[CV 3/5] END bootstrap=False, max_features=0.5, n_estimators=760;, score=0.944 total time=   1.4s
[CV 1/5] END bootstrap=False, max_features=0.5, n_estimators=780;, score=0.949 total time=   1.5s
[CV 4/5] END bootstrap=False, max_features=0.5, n_estimators=790;, score=0.956 total time=   1.5s
[CV 2/5] END bootstrap=False, max_features=0.5, n_estimators=810;, score=0.829 total time=   1.5s
[CV 5/5] END bootstrap=False, max_features=0.5, n_estimators=820;, score=0.934 total time=   1.6s
[CV 3/5] END bootstrap=False, max_features=0.5, n_estimators=840;, score=0.945 total time=   1.6s
[CV 1/5] END bootstrap=False, max_features=0.5, n_estimators=860;, score=0.947 total time=   1.6s
[CV 4/5] END bootstrap=False, max_features=0.5, n_es

,estimator,ExtraTreesRegressor()
,param_grid,"{'bootstrap': [True, False], 'max_features': ['sqrt', 'log2', ...], 'n_estimators': array([ 10, ... 1000, 1010])}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,np.int64(340)


In [48]:
grid.best_params_

{'bootstrap': False, 'max_features': 1.0, 'n_estimators': np.int64(340)}

In [50]:
grid.score(X2_train,Y2_train)
grid.score(X2_test,Y2_test)

0.9923307176879472

In [51]:
#BUILD EXTRA TREE REGRESSOR MODELS FOR GHUSURI DATA
etr_ghusuri = ExtraTreesRegressor(bootstrap = False, max_features = 1.0, n_estimators = 130, random_state=42)
etr_ghusuri.fit(X1_train,Y1_train)


,n_estimators,130
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [53]:
etr_ghusuri.score(X1_train,Y1_train)
etr_ghusuri.score(X1_test,Y1_test)

0.9730995938545608

In [54]:
#BUILD EXTRA TREE REGRESSOR MODELS FOR GHUSURI RBU DATA
etr_ghusuri_frb = ExtraTreesRegressor(bootstrap = False, max_features = 1.0, n_estimators = 340, random_state=42)

In [55]:
etr_ghusuri_frb.fit(X2_train,Y2_train)

,n_estimators,340
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [57]:
etr_ghusuri_frb.score(X2_train,Y2_train)
etr_ghusuri_frb.score(X2_test,Y2_test)

0.9929055474254211


[CV 3/5] END bootstrap=False, max_features=0.7, n_estimators=110;, score=0.981 total time=   0.2s
[CV 1/5] END bootstrap=False, max_features=0.7, n_estimators=130;, score=0.979 total time=   0.3s
[CV 5/5] END bootstrap=False, max_features=0.7, n_estimators=140;, score=0.978 total time=   0.3s
[CV 3/5] END bootstrap=False, max_features=0.7, n_estimators=160;, score=0.980 total time=   0.3s
[CV 5/5] END bootstrap=False, max_features=0.7, n_estimators=170;, score=0.979 total time=   0.3s
[CV 3/5] END bootstrap=False, max_features=0.7, n_estimators=190;, score=0.981 total time=   0.4s
[CV 1/5] END bootstrap=False, max_features=0.7, n_estimators=210;, score=0.979 total time=   0.4s
[CV 4/5] END bootstrap=False, max_features=0.7, n_estimators=220;, score=0.987 total time=   0.5s
[CV 3/5] END bootstrap=False, max_features=0.7, n_estimators=240;, score=0.981 total time=   0.5s
[CV 1/5] END bootstrap=False, max_features=0.7, n_estimators=260;, score=0.979 total time=   0.5s
[CV 5/5] END bootst

In [58]:
#SAVE THE MODELS AS PICKLE FILES
jb.dump(etr_ghusuri, 'etr_ghusuri.pkl')
jb.dump(etr_ghusuri_frb, 'etr_ghusuri_frb.pkl')

['etr_ghusuri_frb.pkl']

In [59]:
#PREDICT PM2.5 FOR GHUSURI DATA
predtrain_b = etr_ghusuri.predict(X1_train)
predtest_b = etr_ghusuri.predict(X1_test)

In [60]:
train_b['ETR_ROI'] = predtrain_b * (train_b['PM2.5'].max() - train_b['PM2.5'].min()) + train_b['PM2.5'].min()
test_b['ETR_ROI'] = predtest_b * (train_b['PM2.5'].max() - train_b['PM2.5'].min()) + train_b['PM2.5'].min()

In [61]:
#PREDICT PM2.5 FOR GHUSURI RBU DATA
predtrain_bf = etr_ghusuri_frb.predict(X2_train)
predtest_bf = etr_ghusuri_frb.predict(X2_test)

In [62]:
train_bf['ETR_ROI'] = predtrain_bf * (train_bf['PM2.5'].max() - train_bf['PM2.5'].min()) + train_bf['PM2.5'].min()
test_bf['ETR_ROI'] = predtest_bf * (train_bf['PM2.5'].max() - train_bf['PM2.5'].min()) + train_bf['PM2.5'].min()


In [63]:
#SAVE AS CSV
#GHUSURI
train_b.to_csv('ghusuri_train.csv',index=False)
test_b.to_csv('ghusuri_test.csv',index=False)


In [64]:
#GHUSURI RBU
train_bf.to_csv('ghusuri_frb_train.csv',index=False)
test_bf.to_csv('ghusuri_frb_test.csv',index=False)

In [65]:
#================MODEL II=============================================

In [66]:
#MERGE TWO CSV
train_ghusuri_df = pd.merge(train_b,train_bf, on='ImageFile',how='inner')
test_ghusuri_df = pd.merge(test_b,test_bf, on='ImageFile',how='inner')

In [67]:
train_ghusuri_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1531 entries, 0 to 1530
Data columns (total 43 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ImageFile         1531 non-null   object 
 1   RH_x              1531 non-null   float64
 2   WS_x              1531 non-null   float64
 3   AT_x              1531 non-null   float64
 4   PM2.5_x           1531 non-null   float64
 5   SSIM_ROI_x        1531 non-null   float64
 6   ERODE_SSIM_ROI_x  1531 non-null   float64
 7   O_SSIM_ROI_x      1531 non-null   float64
 8   Day_x             1531 non-null   int64  
 9   Month_x           1531 non-null   object 
 10  Hour_x            1531 non-null   int64  
 11  Min_x             1531 non-null   int64  
 12  Month_num_x       1531 non-null   int64  
 13  Day_sin_x         1531 non-null   float64
 14  Day_cos_x         1531 non-null   float64
 15  Mon_sin_x         1531 non-null   float64
 16  Mon_cos_x         1531 non-null   float64


In [68]:
#CALCULATE RESIDUAL
train_ghusuri_df['Res'] = train_ghusuri_df['PM2.5_x'] - train_ghusuri_df['ETR_ROI_y']
test_ghusuri_df['Res'] = test_ghusuri_df['PM2.5_x'] - test_ghusuri_df['ETR_ROI_y']

KeyError: 'ETR_ROI_y'

In [69]:
train_ghusuri_df.columns

Index(['ImageFile', 'RH_x', 'WS_x', 'AT_x', 'PM2.5_x', 'SSIM_ROI_x',
       'ERODE_SSIM_ROI_x', 'O_SSIM_ROI_x', 'Day_x', 'Month_x', 'Hour_x',
       'Min_x', 'Month_num_x', 'Day_sin_x', 'Day_cos_x', 'Mon_sin_x',
       'Mon_cos_x', 'Hour_sin_x', 'Hour_cos_x', 'Min_sin_x', 'Min_cos_x',
       'PM2.5_Norm_x', 'RH_y', 'WS_y', 'AT_y', 'PM2.5_y', 'SSIM_ROI_y',
       'ERODE_SSIM_ROI_y', 'O_SSIM_ROI_y', 'Day_y', 'Month_y', 'Hour_y',
       'Min_y', 'Month_num_y', 'Day_sin_y', 'Day_cos_y', 'Mon_sin_y',
       'Mon_cos_y', 'Hour_sin_y', 'Hour_cos_y', 'Min_sin_y', 'Min_cos_y',
       'PM2.5_Norm_y'],
      dtype='object')

In [70]:
train_b.columns


Index(['ImageFile', 'RH', 'WS', 'AT', 'PM2.5', 'SSIM_ROI', 'ERODE_SSIM_ROI',
       'O_SSIM_ROI', 'Day', 'Month', 'Hour', 'Min', 'Month_num', 'Day_sin',
       'Day_cos', 'Mon_sin', 'Mon_cos', 'Hour_sin', 'Hour_cos', 'Min_sin',
       'Min_cos', 'PM2.5_Norm'],
      dtype='object')

In [71]:
train_bf.columns

Index(['ImageFile', 'RH', 'WS', 'AT', 'PM2.5', 'SSIM_ROI', 'ERODE_SSIM_ROI',
       'O_SSIM_ROI', 'Day', 'Month', 'Hour', 'Min', 'Month_num', 'Day_sin',
       'Day_cos', 'Mon_sin', 'Mon_cos', 'Hour_sin', 'Hour_cos', 'Min_sin',
       'Min_cos', 'PM2.5_Norm'],
      dtype='object')

In [66]:
train_b=pd.read_csv("ghusuri_train.csv")
test_b=pd.read_csv("ghusuri_test.csv")

In [67]:
train_b.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2090 entries, 0 to 2089
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ImageFile       2090 non-null   object 
 1   RH              2090 non-null   float64
 2   WS              2090 non-null   float64
 3   AT              2090 non-null   float64
 4   PM2.5           2090 non-null   float64
 5   SSIM_ROI        2090 non-null   float64
 6   ERODE_SSIM_ROI  2090 non-null   float64
 7   O_SSIM_ROI      2090 non-null   float64
 8   Day             2090 non-null   int64  
 9   Month           2090 non-null   object 
 10  Hour            2090 non-null   int64  
 11  Min             2090 non-null   int64  
 12  Month_num       2090 non-null   int64  
 13  Day_sin         2090 non-null   float64
 14  Day_cos         2090 non-null   float64
 15  Mon_sin         2090 non-null   float64
 16  Mon_cos         2090 non-null   float64
 17  Hour_sin        2090 non-null   f

In [68]:
train_bf=pd.read_csv("ghusuri_frb_train.csv")
test_bf=pd.read_csv("ghusuri_frb_test.csv")

In [69]:
test_bf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 526 entries, 0 to 525
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ImageFile       526 non-null    object 
 1   RH              526 non-null    float64
 2   WS              526 non-null    float64
 3   AT              526 non-null    float64
 4   PM2.5           526 non-null    float64
 5   SSIM_ROI        526 non-null    float64
 6   ERODE_SSIM_ROI  526 non-null    float64
 7   O_SSIM_ROI      526 non-null    float64
 8   Day             526 non-null    int64  
 9   Month           526 non-null    object 
 10  Hour            526 non-null    int64  
 11  Min             526 non-null    int64  
 12  Month_num       526 non-null    int64  
 13  Day_sin         526 non-null    float64
 14  Day_cos         526 non-null    float64
 15  Mon_sin         526 non-null    float64
 16  Mon_cos         526 non-null    float64
 17  Hour_sin        526 non-null    flo

In [70]:
train_ghusuri_df=pd.merge(train_b,train_bf,on='ImageFile',how='inner')
test_ghusuri_df=pd.merge(test_b,test_bf,on='ImageFile',how='inner')

In [71]:
train_ghusuri_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1666 entries, 0 to 1665
Data columns (total 45 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ImageFile         1666 non-null   object 
 1   RH_x              1666 non-null   float64
 2   WS_x              1666 non-null   float64
 3   AT_x              1666 non-null   float64
 4   PM2.5_x           1666 non-null   float64
 5   SSIM_ROI_x        1666 non-null   float64
 6   ERODE_SSIM_ROI_x  1666 non-null   float64
 7   O_SSIM_ROI_x      1666 non-null   float64
 8   Day_x             1666 non-null   int64  
 9   Month_x           1666 non-null   object 
 10  Hour_x            1666 non-null   int64  
 11  Min_x             1666 non-null   int64  
 12  Month_num_x       1666 non-null   int64  
 13  Day_sin_x         1666 non-null   float64
 14  Day_cos_x         1666 non-null   float64
 15  Mon_sin_x         1666 non-null   float64
 16  Mon_cos_x         1666 non-null   float64


In [72]:
test_ghusuri_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 45 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ImageFile         146 non-null    object 
 1   RH_x              146 non-null    float64
 2   WS_x              146 non-null    float64
 3   AT_x              146 non-null    float64
 4   PM2.5_x           146 non-null    float64
 5   SSIM_ROI_x        146 non-null    float64
 6   ERODE_SSIM_ROI_x  146 non-null    float64
 7   O_SSIM_ROI_x      146 non-null    float64
 8   Day_x             146 non-null    int64  
 9   Month_x           146 non-null    object 
 10  Hour_x            146 non-null    int64  
 11  Min_x             146 non-null    int64  
 12  Month_num_x       146 non-null    int64  
 13  Day_sin_x         146 non-null    float64
 14  Day_cos_x         146 non-null    float64
 15  Mon_sin_x         146 non-null    float64
 16  Mon_cos_x         146 non-null    float64
 1

In [73]:
#CALCULATE RESIDUAL
train_ghusuri_df['Res'] = train_ghusuri_df['PM2.5_x'] - train_ghusuri_df['ETR_ROI_y']
test_ghusuri_df['Res'] = test_ghusuri_df['PM2.5_x'] - test_ghusuri_df['ETR_ROI_y']

In [74]:
#NORMALIZE RESIDUAL
train_ghusuri_df['Res_Norm'] = (train_ghusuri_df['Res'] - train_ghusuri_df['Res'].min())/(train_ghusuri_df['Res'].max() - train_ghusuri_df['Res'].min())
test_ghusuri_df['Res_Norm'] = (test_ghusuri_df['Res'] - train_ghusuri_df['Res'].min())/(train_ghusuri_df['Res'].max() - train_ghusuri_df['Res'].min())

In [75]:
#CREATE X LIST FOR GHUSURI X GHUSURI RBU DATA
X_train_res = train_ghusuri_df[['SSIM_ROI_x','ERODE_SSIM_ROI_x','Day_sin_x','Day_cos_x','Mon_sin_x','Mon_cos_x','Hour_sin_x','Hour_cos_x','PM2.5_Norm_y']]
X_test_res = test_ghusuri_df[['SSIM_ROI_x','ERODE_SSIM_ROI_x','Day_sin_x','Day_cos_x','Mon_sin_x','Mon_cos_x','Hour_sin_x','Hour_cos_x','PM2.5_Norm_y']]

In [76]:
#CREATE Y LIST FOR GHUSURI X GHUSURI RBU DATA
Y_train_res = train_ghusuri_df['Res_Norm']
Y_test_res = test_ghusuri_df['Res_Norm']

In [77]:
#GRID SEARCH
grid = GridSearchCV(ExtraTreesRegressor(),param_grid=param_grid,refit=True,verbose=3,n_jobs=-1)

In [78]:
grid.fit(X_train_res,Y_train_res)

Fitting 5 folds for each of 1616 candidates, totalling 8080 fits
[CV 1/5] END bootstrap=True, max_features=sqrt, n_estimators=20;, score=0.861 total time=   0.0s
[CV 4/5] END bootstrap=True, max_features=sqrt, n_estimators=20;, score=0.855 total time=   0.0s
[CV 5/5] END bootstrap=True, max_features=sqrt, n_estimators=20;, score=0.854 total time=   0.0s
[CV 1/5] END bootstrap=True, max_features=sqrt, n_estimators=30;, score=0.876 total time=   0.1s
[CV 1/5] END bootstrap=True, max_features=sqrt, n_estimators=40;, score=0.871 total time=   0.1s
[CV 1/5] END bootstrap=True, max_features=sqrt, n_estimators=60;, score=0.876 total time=   0.1s
[CV 5/5] END bootstrap=True, max_features=sqrt, n_estimators=70;, score=0.884 total time=   0.1s
[CV 4/5] END bootstrap=True, max_features=sqrt, n_estimators=90;, score=0.880 total time=   0.1s
[CV 3/5] END bootstrap=True, max_features=sqrt, n_estimators=110;, score=0.862 total time=   0.1s
[CV 4/5] END bootstrap=True, max_features=sqrt, n_estimators=

/opt/anaconda3/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



[CV 3/5] END bootstrap=True, max_features=log2, n_estimators=400;, score=0.876 total time=   0.5s
[CV 5/5] END bootstrap=True, max_features=log2, n_estimators=410;, score=0.889 total time=   0.5s
[CV 2/5] END bootstrap=True, max_features=log2, n_estimators=430;, score=0.657 total time=   0.6s
[CV 5/5] END bootstrap=True, max_features=log2, n_estimators=440;, score=0.886 total time=   0.5s
[CV 2/5] END bootstrap=True, max_features=log2, n_estimators=460;, score=0.646 total time=   0.6s
[CV 5/5] END bootstrap=True, max_features=log2, n_estimators=470;, score=0.888 total time=   0.6s
[CV 3/5] END bootstrap=True, max_features=log2, n_estimators=490;, score=0.875 total time=   0.7s
[CV 1/5] END bootstrap=True, max_features=log2, n_estimators=510;, score=0.898 total time=   0.7s
[CV 4/5] END bootstrap=True, max_features=log2, n_estimators=520;, score=0.883 total time=   0.7s
[CV 1/5] END bootstrap=True, max_features=log2, n_estimators=540;, score=0.897 total time=   0.7s
[CV 4/5] END bootst

,estimator,ExtraTreesRegressor()
,param_grid,"{'bootstrap': [True, False], 'max_features': ['sqrt', 'log2', ...], 'n_estimators': array([ 10, ... 1000, 1010])}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,np.int64(50)



[CV 3/5] END bootstrap=False, max_features=0.7, n_estimators=510;, score=0.922 total time=   1.0s
[CV 1/5] END bootstrap=False, max_features=0.7, n_estimators=530;, score=0.923 total time=   1.0s
[CV 4/5] END bootstrap=False, max_features=0.7, n_estimators=540;, score=0.949 total time=   1.0s
[CV 2/5] END bootstrap=False, max_features=0.7, n_estimators=560;, score=0.838 total time=   1.0s
[CV 5/5] END bootstrap=False, max_features=0.7, n_estimators=570;, score=0.917 total time=   1.0s
[CV 3/5] END bootstrap=False, max_features=0.7, n_estimators=590;, score=0.914 total time=   1.1s
[CV 1/5] END bootstrap=False, max_features=0.7, n_estimators=610;, score=0.924 total time=   1.1s
[CV 4/5] END bootstrap=False, max_features=0.7, n_estimators=620;, score=0.947 total time=   1.1s
[CV 2/5] END bootstrap=False, max_features=0.7, n_estimators=640;, score=0.848 total time=   1.2s
[CV 5/5] END bootstrap=False, max_features=0.7, n_estimators=650;, score=0.914 total time=   1.2s
[CV 3/5] END bootst

In [79]:
grid.best_params_

{'bootstrap': False, 'max_features': 1.0, 'n_estimators': np.int64(50)}

In [80]:
grid.score(X_train_res,Y_train_res)

1.0

In [81]:
grid.score(X_test_res,Y_test_res)

0.9817370635501677

In [83]:
#EXTRA TREE REGRESSOR
etr_ghusuri_res = ExtraTreesRegressor(bootstrap = False, max_features = 1.0, n_estimators = 50, random_state=42)

In [84]:
etr_ghusuri_res.fit(X_train_res,Y_train_res)

,n_estimators,50
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,False
,oob_score,False


In [85]:
etr_ghusuri_res.score(X_train_res,Y_train_res)

1.0

In [86]:
etr_ghusuri_res.score(X_test_res,Y_test_res)

0.9854060300636746

In [87]:
#SAVE AS PICKLE FILES
jb.dump(etr_ghusuri_res, 'etr_ghusuri_res.pkl')

['etr_ghusuri_res.pkl']

In [88]:
#PREDICT RESIDUALS
predtrain=etr_ghusuri_res.predict(X_train_res)
predtest=etr_ghusuri_res.predict(X_test_res)

In [89]:
train_ghusuri_df['ETR_Res'] = predtrain * (train_ghusuri_df['Res'].max() - train_ghusuri_df['Res'].min()) + train_ghusuri_df['Res'].min()
test_ghusuri_df['ETR_Res'] = predtest * (train_ghusuri_df['Res'].max() - train_ghusuri_df['Res'].min()) + train_ghusuri_df['Res'].min()

In [90]:
#CALCULATE FINAL PREDICTED PM2.5
train_ghusuri_df['Pred_PM2.5'] = train_ghusuri_df['ETR_ROI_y'] + train_ghusuri_df['ETR_Res']
test_ghusuri_df['Pred_PM2.5'] = test_ghusuri_df['ETR_ROI_y'] + test_ghusuri_df['ETR_Res']


In [91]:
#CHECKING R^2
r2_score(train_ghusuri_df['PM2.5_x'],train_ghusuri_df['Pred_PM2.5'])

1.0

In [92]:
r2_score(test_ghusuri_df['PM2.5_x'],test_ghusuri_df['Pred_PM2.5'])

0.9888580207967311

In [93]:
#SAVE AS CSV
train_ghusuri_df.to_csv('final_train_ghusuri.csv',index=False)
test_ghusuri_df.to_csv('final_test_ghusuri.csv',index=False)
